# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 · Housing Financial Vulnerability Score (HFVS)
### Phase 2 — Data Cleaning & Preparation (Revised: ≤50-column output)

| | |
|---|---|
| **Student** | Valerie Jerono |
| **Reg. No.** | 222331 |
| **Supervisor** | Prof. Jacob Ong'ala |
| **Institution** | Strathmore University — iLabAfrica |
| **Date** | June 2026 |

---

### What this notebook does

Takes `master_frame.parquet` — **21,347 households × 443 columns** — and produces a principled,
analysis-ready working frame of **≤50 columns**.

**Key design principle:** Every raw column that exists only to engineer a derived feature is used
transiently during engineering and then dropped. Only the final analytical variable survives into
the output frame. No `_imputed` flag columns are carried forward — imputation decisions are
documented in audit tables, not encoded as extra columns.

**Pipeline (in order):**

| Stage | Action |
|-------|--------|
| 0 | Environment setup · paths · county map |
| 1 | Column rename (KNBS codes → semantic names) |
| 2 | KNBS sentinel-code decoding (−1, 96/98/99 → NaN) |
| 3 | Column audit (missingness · module) |
| 4 | Structural drops (free-text · admin · constants · >90% missing) |
| 5 | Structural missingness encoding (k_ · l_ · lp_) |
| 6 | Feature engineering (all 5 pillars) |
| 7 | Tenure-stratified imputation (residual NaN only, no flag columns kept) |
| 8 | KES outlier treatment (Winsorise at p99 non-zero) |
| 9 | Consistency checks |
| 10 | **Final column selection → exactly the 47-column analytical frame** |
| 11 | Pillar map integrity check + anti-leakage assertion |
| 12 | Save cleaned frame + audit tables |

> **Target output:** `master_frame_clean.parquet` — 21,347 households × ≤50 columns


## 0 · Environment Setup

In [ ]:
# ── 0.1  Mount Google Drive (Colab) or fall back to local CWD ──────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Running in Google Colab — Drive mounted.')
except Exception:
    IN_COLAB = False
    print('Running locally — using current working directory.')


In [ ]:
# ── 0.2  Install dependencies ────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'scipy', 'statsmodels'],
    check=False
)


In [ ]:
# ── 0.3  Core imports ────────────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(8301)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 80)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})
TEAL = '#00695C'; RED = '#B71C1C'; AMBER = '#E65100'
BLUE = '#1565C0'; GRAY = '#546E7A'

print('Imports loaded. NumPy', np.__version__, '| Pandas', pd.__version__)


In [ ]:
# ── 0.4  Paths ───────────────────────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation') if IN_COLAB else Path.cwd()
PQ    = DRIVE / 'data' / 'parquet'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301_clean'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301_clean'
for p in [FIGS, TABS]:
    p.mkdir(parents=True, exist_ok=True)
print(f'BASE → {DRIVE}')


In [ ]:
# ── 0.5  County map (KNBS: 1–47) ─────────────────────────────────────────────
COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'County map: {len(COUNTY_MAP)} counties.')


## 1 · Column Rename Map (KNBS codes → semantic names)

Applied before sentinel decoding so all subsequent code works with readable names.
Only columns present in the frame are renamed — the rest are silently skipped.

> **Why rename first?** Every drop and imputation decision below references semantic
> names, not KNBS codes. Renaming first makes every subsequent step auditable.


In [ ]:
# ── 1.1  Full rename map ─────────────────────────────────────────────────────
RENAME_MAP = {
    # Admin / geo / weights
    'interview__key': 'hh_id',        'interview__id': 'hh_uuid',
    'a01':   'county_code',           'countycode':    'county_code_str',
    'a07_1': 'urban_rural',           'serial':        'hh_serial',
    'hhweight': 'hh_weight',          'a12':           'interview_result',
    'tag':   'survey_tag',            'rnd':           'sample_round',
    'selectedage': 'selected_respondent_age',

    # Water & sanitation
    'c01_1': 'water_src_main',        'c01_2': 'water_collect_method',
    'c01_3': 'water_treated',         'c01_4': 'water_dist_mins',
    'c03':   'n_toilet_facilities',   'c04':   'toilet_type',
    'c05':   'has_handwash_facility', 'c08':   'has_bathhouse',
    'c14_1': 'spend_water_kes',

    # Energy
    'c10':   'lighting_src',          'c10_2': 'electricity_hrs_day',
    'c10_3': 'electricity_supply_type','c10_4': 'electricity_conn_type',
    'c11':   'cooking_fuel',          'c12':   'cooking_stove_type',
    'c14_2': 'spend_electricity_kes', 'c14_3': 'spend_energy_other_kes',

    # Assets
    'c13__2': 'owns_mobile',          'c13__4': 'owns_computer',
    'c13__6': 'owns_vehicle',         'internet': 'has_internet',

    # Dwelling (household module)
    'd01':    'hh_tenure_type',       'd17':   'dwelling_ownership_doc',
    'duration': 'tenancy_duration_cat','year_occ': 'year_occupied_cat',

    # Expenditure (g01a–k)
    'g01a': 'spend_food_kes',         'g01b': 'spend_clothing_kes',
    'g01c': 'spend_education_kes',    'g01d': 'spend_health_kes',
    'g01e': 'spend_transport_kes',    'g01f': 'spend_comms_kes',
    'g01g': 'spend_recreation_kes',   'g01h': 'spend_housing_kes',
    'g01i': 'spend_energy_kes',       'g01j': 'spend_other_kes',
    'g01k': 'spend_remittances_kes',
    'g02':  'pays_rent',              'g02_1': 'rent_monthly_kes',
    'g03':  'tenure_type',            'g04':   'owns_other_property',

    # Housing perception (h01–h11)
    'h01': 'perc_structure', 'h02': 'perc_roof',     'h03': 'perc_walls',
    'h04': 'perc_floor',     'h05': 'perc_ventilation','h06': 'perc_lighting',
    'h07': 'perc_water',     'h08': 'perc_sanitation','h09': 'perc_waste',
    'h10': 'perc_security',  'h11': 'perc_overall',

    # Land ownership
    'i00': 'owns_land',

    # Environment & hazards (e module)
    'e05': 'dist_to_market_mins',     'e06': 'flood_exposure',
    'e07': 'landslide_exposure',      'e08': 'other_hazard_exposure',
    'e09__3': 'hazard_displaced',

    # Tenure & mobility (j module)
    'j04_1': 'is_owner_occupier',     'j04_2': 'tenure_is_informal',
    'j05':   'has_title_doc',         'j09':   'housing_cost_burden',
    'j10':   'missed_payment',        'j11':   'eviction_risk',
    'j12_1': 'yrs_in_dwelling',       'j13':   'satisfied_tenure',
    'j14':   'wants_to_own',

    # Rental module (k)
    'k02': 'has_written_lease',       'k05': 'rent_actual_kes',
    'k21': 'rent_arrears',

    # Owned dwelling (l module)
    'l08': 'dwelling_yr_built',       'l15': 'mortgage_repayment_kes',

    # Derived / pre-computed
    'sf':   'is_slum',                'pln': 'settlement_plan_status',

    # Dwelling file aggregates
    'dw_type': 'dw_type',             'dw_wall_material': 'dw_wall_mat',
    'dw_roof_material': 'dw_roof_mat','dw_floor_material': 'dw_floor_mat',
    'dw_n_rooms': 'dw_rooms',         'dw_floor_area_m2': 'dw_area_m2',
    'dw_n_bedrooms': 'dw_bedrooms',   'dw_approved': 'dw_approved',
    'dw_planning_ok': 'dw_has_planning','dw_hazard_zone': 'dw_in_hazard_zone',

    # Individual aggregates
    'hh_size': 'hh_size',             'hhh_sex': 'hh_head_sex',
    'any_disability': 'has_disability','max_edu_isced': 'max_edu_isced',
    'mean_age': 'mean_age',            'dependency_ratio': 'dependency_ratio',

    # Land parcel aggregates
    'lp_n_parcels': 'lp_n_parcels',   'lp_primary_tenure': 'lp_tenure_type',
    'lp_has_title': 'lp_has_title',   'lp_primary_use': 'lp_land_use',
    'lp_any_dispute': 'lp_has_dispute','lp_any_registered': 'lp_is_registered',
    'lp_any_collateral': 'lp_used_as_collateral',

    # Mortgage / loan / financier
    'mort_interest_rate': 'mort_rate',
}
INVERSE_MAP = {v: k for k, v in RENAME_MAP.items()}
print(f'RENAME_MAP: {len(RENAME_MAP)} entries defined.')


## 2 · Load, Rename & Sentinel Decode

KNBS SurveySolutions encodes non-responses as numeric sentinels that must become NaN
before any statistics or imputation. **Protected columns** (weights, binary indicators,
county code) are excluded from sentinel decoding.


In [ ]:
# ── 2.1  Load master_frame.parquet ───────────────────────────────────────────
candidates = [
    PQ / 'master_frame.parquet',
    Path('master_frame.parquet'),
    Path('/content/master_frame.parquet'),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Cannot find master_frame.parquet. '
        'Place it in data/parquet/, the CWD, or /content/.'
    )
df = pd.read_parquet(DATA_PATH)
print(f'Loaded   → {DATA_PATH}')
print(f'Shape    → {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory   → {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

assert df.duplicated().sum() == 0, 'Duplicated rows found — resolve in master frame build.'
print('No duplicated rows. ✓')


In [ ]:
# ── 2.2  Apply rename map ────────────────────────────────────────────────────
rename_now = {raw: sem for raw, sem in RENAME_MAP.items() if raw in df.columns}
df = df.rename(columns=rename_now)
print(f'Renamed {len(rename_now)} columns. Frame: {df.shape}')


In [ ]:
# ── 2.3  Sentinel decoding ───────────────────────────────────────────────────
# Columns where −1, 96/98/99, 996/998/999 are VALID values (not sentinels)
PROTECTED_FROM_SENTINEL = {
    'county_code', 'hh_weight', 'urban_rural', 'hh_id',
    'owns_mobile', 'has_internet', 'has_disability',
    'dw_approved', 'dw_has_planning', 'dw_in_hazard_zone',
    'lp_has_title', 'lp_has_dispute', 'lp_is_registered', 'lp_used_as_collateral',
    'has_handwash_facility', 'pays_rent',
}

SENTINELS = {-1, 96, 98, 99, 996, 997, 998, 999}
sentinel_log = []
for c in df.select_dtypes(include='number').columns:
    if c in PROTECTED_FROM_SENTINEL:
        continue
    mask = df[c].isin(SENTINELS)
    n = mask.sum()
    if n > 0:
        df.loc[mask, c] = np.nan
        sentinel_log.append({'column': c, 'n_decoded': n})

sentinel_df = pd.DataFrame(sentinel_log).sort_values('n_decoded', ascending=False)
sentinel_df.to_csv(TABS / '01_sentinel_decoding_log.csv', index=False)
print(f'Columns with sentinels decoded : {len(sentinel_df)}')
print(f'Total cells set to NaN        : {sentinel_df["n_decoded"].sum():,}')
print()
print('Top 10 columns by cells decoded:')
print(sentinel_df.head(10).to_string(index=False))


## 3 · Column Audit

A quick missingness inventory **after** sentinel decoding — so percentages reflect true
absence, not hidden sentinel codes. This drives drop and imputation decisions.


In [ ]:
# ── 3.1  Full column audit ───────────────────────────────────────────────────
audit_rows = []
for c in df.columns:
    s = df[c]
    pct_miss = s.isna().mean() * 100
    audit_rows.append({
        'column':      c,
        'dtype':       str(s.dtype),
        'pct_missing': round(pct_miss, 2),
        'n_unique':    s.nunique(dropna=True),
    })

audit = pd.DataFrame(audit_rows).set_index('column')
audit.to_csv(TABS / '02_full_column_audit.csv')

def miss_tier(p):
    if p == 0:   return '0_Complete'
    if p <= 25:  return '1_Low'
    if p <= 60:  return '2_Moderate'
    return               '3_High'

audit['miss_tier'] = audit['pct_missing'].apply(miss_tier)

print(f'Total columns audited: {len(audit)}')
print()
print('Missingness tier breakdown:')
print(audit['miss_tier'].value_counts().sort_index().to_string())
print()
print(f'Constant columns (n_unique ≤ 1): {(audit["n_unique"] <= 1).sum()}')


## 4 · Structural Drops

Drop columns that carry zero analytical information. These are dropped *before* any
imputation or engineering so no resources are wasted on useless columns.

| Category | Rationale |
|----------|-----------|
| Free-text "other/specify" fields | Open strings; not codeable |
| Admin / survey-logistics identifiers | Non-analytical |
| Constant columns (≤1 unique value) | Zero information content |
| >90% missing (non-structural) | Too sparse to use or impute |

> **Note:** Columns in the five HFVS pillars that are legitimately high-missing
> (k_*, l_*, lp_* at 55–68%) are handled in Stage 5, not dropped here.


In [ ]:
# ── 4.1  Define columns needed transiently for engineering ──────────────────
# These are kept through engineering (Stage 6), then dropped in Stage 10.
TRANSIENT_FOR_ENGINEERING = {
    # Needed to build total_exp → then dropped
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
    'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
    'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
    'spend_other_kes', 'spend_remittances_kes',
    # Needed to build rent_burden_ratio → then dropped
    'rent_actual_kes', 'rent_monthly_kes', 'pays_rent',
    # Needed to build obj_quality_score → then dropped
    'dw_wall_mat', 'dw_roof_mat', 'dw_floor_mat',
    # Needed to build dw_age_yrs → then dropped
    'dwelling_yr_built',
    # Needed to build util_burden_ratio → then dropped
    'spend_water_kes', 'spend_electricity_kes',
    # Needed to build persons_per_room → then dropped
    'dw_rooms',
    # Needed to build triple_exposed → then dropped
    'has_title_doc',   # kept in final anyway
    # Needed for is_renter / is_owner flags
    'tenure_type',     # kept in final anyway
    # perc_* individual components → only perc_overall survives into final
    'perc_structure', 'perc_roof', 'perc_walls', 'perc_floor',
    'perc_ventilation', 'perc_lighting', 'perc_water', 'perc_sanitation',
    'perc_waste', 'perc_security',
    # Needed for lp-encoding
    'owns_land',       # kept in final anyway
}

# ── 4.2  The definitive final-frame column set (47 columns) ──────────────────
# Nothing outside this set will survive into the output parquet.
# All engineering creates columns in this set; all others are dropped at Stage 10.
FINAL_47 = {
    # IDs / weights
    'hh_id', 'county_code', 'county_name', 'hh_weight',
    # Pillar 1 — Financial Stress (all engineered or validated)
    'total_exp', 'rent_burden_ratio', 'rent_burdened',
    'housing_cost_burden', 'missed_payment', 'eviction_risk',
    # Pillar 2 — Physical Quality
    'obj_quality_score', 'perc_overall', 'dw_type', 'dw_rooms',
    'persons_per_room', 'is_overcrowded', 'dw_age_yrs',
    # Pillar 3 — Tenure Security
    'tenure_type', 'has_title_doc', 'owns_land', 'lp_has_title',
    'lp_has_dispute', 'satisfied_tenure',
    # Pillar 4 — Physical Hazard
    'flood_exposure', 'landslide_exposure', 'triple_exposed',
    'dw_in_hazard_zone', 'hazard_displaced',
    # Pillar 5 — Utility Deprivation
    'water_src_main', 'toilet_type', 'cooking_fuel', 'lighting_src',
    'has_handwash_facility', 'util_burden_ratio',
    # Demographics / proxy features
    'hh_size', 'hh_head_sex', 'has_disability', 'max_edu_isced',
    'dependency_ratio', 'urban_rural', 'owns_mobile', 'has_internet',
    'is_renter', 'is_owner',
    # Context
    'is_slum', 'settlement_plan_status', 'water_treated',
}
print(f'FINAL_47 target columns: {len(FINAL_47)}')

# Columns to preserve through all intermediate steps
KEEP_THROUGH_PIPELINE = FINAL_47 | TRANSIENT_FOR_ENGINEERING


In [ ]:
# ── 4.3  Run structural drops ────────────────────────────────────────────────
import re

FREE_TEXT_RX = re.compile(r'(other_text|other_specify|_other\d*$|_other$|_text$|_specify$)', re.I)
free_text_cols = [c for c in df.columns
                  if FREE_TEXT_RX.search(c) and c not in KEEP_THROUGH_PIPELINE]

admin_drop = [c for c in [
    'survey_tag', 'sample_round', 'selected_respondent_age',
    'county_code_str', 'hh_serial', 'interview_result', 'hh_uuid',
    'dwelling_yr_surveyed', 'yr_last_renovated', 'mean_age',
    'n_female', 'n_children', 'n_elderly', 'n_working_age',
    'dw_units_count', 'building_floor_cat', 'n_hh_in_building',
] if c in df.columns and c not in KEEP_THROUGH_PIPELINE]

constant_cols = [c for c in df.columns
                 if df[c].nunique(dropna=True) <= 1 and c not in KEEP_THROUGH_PIPELINE]

near_empty_cols = audit.index[
    (audit['pct_missing'] > 90) & (~audit.index.isin(KEEP_THROUGH_PIPELINE))
].tolist()

# Drop everything NOT in our keep set (the most direct approach)
cols_to_drop = [c for c in df.columns if c not in KEEP_THROUGH_PIPELINE]
df = df.drop(columns=cols_to_drop)

print(f'Columns dropped (not needed): {len(cols_to_drop)}')
print(f'Shape after structural drops: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Remaining columns: {sorted(df.columns.tolist())}')


## 5 · Structural Missingness Encoding

Three blocks of columns have high missingness that is **not random** — it encodes
"not applicable" for households correctly routed away from that module.

| Block | Approx miss% | Root cause | Treatment |
|-------|-------------|------------|-----------|
| Rental `k_*` | ~68% | Tenants only | Categorical → −1 (not applicable) |
| Owned `l_*` | ~55% | Owner-occupiers only | Same |
| Land parcel `lp_*` | ~55% | `owns_land == 1` only | Binary → 0 for landless HHs |

> **Why −1, not NaN?** For categorical columns, −1 is an explicit "not applicable"
> category that survives ordinal encoding without introducing false imputed values.
> For continuous KES columns, NaN is preserved (subpopulation analysis applies them).


In [ ]:
# ── 5.1  Tenure flags (needed for imputation stratification) ─────────────────
if 'pays_rent' in df.columns:
    df['is_renter'] = df['pays_rent'].eq(1).astype(int)
    print(f'is_renter=1 : {df["is_renter"].sum():,} ({df["is_renter"].mean():.1%})')
if 'tenure_type' in df.columns:
    df['is_owner'] = df['tenure_type'].eq(1).astype(int)
    print(f'is_owner=1  : {df["is_owner"].sum():,} ({df["is_owner"].mean():.1%})')


In [ ]:
# ── 5.2  Land parcel (lp_*) — landless households → 0 ───────────────────────
# For owns_land == 0: binary lp_ columns = 0 (no parcels; this is real information,
# not missing data). n_parcels also = 0 for landless households.
if 'owns_land' in df.columns:
    landless = df['owns_land'].eq(0)
    for c in ['lp_has_title', 'lp_has_dispute', 'lp_is_registered', 'lp_used_as_collateral']:
        if c in df.columns:
            df.loc[landless, c] = df.loc[landless, c].fillna(0)
    print(f'Landless (owns_land=0): {landless.sum():,} — lp_ binaries set to 0 ✓')
    print(f'lp_has_title still NaN (among landowners): '
          f'{df.loc[~landless, "lp_has_title"].isna().sum() if "lp_has_title" in df.columns else "N/A"}')


In [ ]:
# ── 5.3  Rental module — structural NaN → -1 (not applicable) ───────────────
# Only for categorical rental columns in the final keep set.
# rent_actual_kes is continuous → kept as NaN (renter subpopulation analysis).
RENTAL_CATEGORICAL = [c for c in ['has_written_lease', 'rent_arrears'] if c in df.columns]
for c in RENTAL_CATEGORICAL:
    n_miss = df[c].isna().sum()
    if n_miss > 0:
        df[c] = df[c].fillna(-1)
        print(f'{c}: {n_miss:,} NaN → -1 (not applicable)')


## 6 · Feature Engineering (All Five Pillars)

All derived features are computed here. The transient raw columns used only as
engineering inputs are **dropped at Stage 10**, keeping the frame lean.

| Feature | Formula | Pillar |
|---------|---------|--------|
| `total_exp` | Σ(spend_food … spend_remittances) | 1 — Financial Stress |
| `rent_burden_ratio` | rent_actual / (total_exp − spend_housing) | 1 |
| `rent_burdened` | rent_burden_ratio > 0.30 (SDG 11 threshold) | 1 |
| `util_burden_ratio` | (spend_water + spend_electricity) / total_exp | 5 — Utility |
| `obj_quality_score` | Mean(wall_score, roof_score, floor_score) | 2 — Physical Quality |
| `persons_per_room` | hh_size / dw_rooms | 2 |
| `is_overcrowded` | persons_per_room > 2 (WHO threshold) | 2 |
| `dw_age_yrs` | 2024 − dwelling_yr_built | 2 |
| `triple_exposed` | flood > 0 AND informal structure AND no title | 4 — Hazard |
| `county_name` | COUNTY_MAP lookup | Geography |


In [ ]:
# ── 6.1  Total expenditure (income proxy) ────────────────────────────────────
EXP_COLS = [
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
    'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
    'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
    'spend_other_kes', 'spend_remittances_kes',
]
present_exp = [c for c in EXP_COLS if c in df.columns]
if len(present_exp) >= 8:
    df['total_exp'] = df[present_exp].sum(axis=1, min_count=8)
    print(f'total_exp: {len(present_exp)}/11 components found.')
    print(f'  Median : KES {df["total_exp"].median():,.0f}/month')
    print(f'  Missing: {df["total_exp"].isna().sum()}')
else:
    raise ValueError(f'Only {len(present_exp)} expenditure columns found — cannot build total_exp.')


In [ ]:
# ── 6.2  Rent burden ratio (renter subpopulation) ────────────────────────────
if {'rent_actual_kes', 'total_exp', 'spend_housing_kes', 'is_renter'}.issubset(df.columns):
    renter_mask = df['is_renter'] == 1
    denom = (df['total_exp'] - df['spend_housing_kes']).clip(lower=1)
    df['rent_burden_ratio'] = np.nan
    df.loc[renter_mask, 'rent_burden_ratio'] = (
        df.loc[renter_mask, 'rent_actual_kes'] / denom.loc[renter_mask]
    ).clip(upper=1.5)
    df['rent_burdened'] = np.nan
    df.loc[renter_mask, 'rent_burdened'] = (
        df.loc[renter_mask, 'rent_burden_ratio'] > 0.30
    ).astype(float)
    n_b = df['rent_burdened'].eq(1).sum()
    n_r = renter_mask.sum()
    print(f'Renter households      : {n_r:,}')
    print(f'Rent-burdened (>30%)   : {n_b:,} / {n_r:,} = {n_b/n_r:.1%}')
else:
    print('WARNING: rent burden columns not all present — check rename map.')


In [ ]:
# ── 6.3  Utility burden ratio ────────────────────────────────────────────────
if {'spend_water_kes', 'spend_electricity_kes', 'total_exp'}.issubset(df.columns):
    util = df['spend_water_kes'].fillna(0) + df['spend_electricity_kes'].fillna(0)
    df['util_burden_ratio'] = (util / df['total_exp'].replace(0, np.nan)).clip(upper=1.0)
    print(f'util_burden_ratio: median={df["util_burden_ratio"].median():.3f}, '
          f'mean={df["util_burden_ratio"].mean():.3f}')


In [ ]:
# ── 6.4  Objective structural quality score ──────────────────────────────────
# Materials coded 0 (worst) → 1 (best); KHS codebook mapping.
WALL_SCORE  = {4:0.00, 5:0.00, 3:0.25, 6:0.25, 2:0.50, 8:0.50,
               1:0.75, 9:0.75, 12:1.00, 7:1.00, -1:np.nan}
ROOF_SCORE  = {3:0.00, 4:0.00, 2:0.33, 5:0.50, 1:0.67, 8:0.67, 6:0.83, 7:1.00, -1:np.nan}
FLOOR_SCORE = {3:0.00, 4:0.00, 5:0.25, 1:0.50, 2:1.00, 6:1.00, -1:np.nan}

_comp_scores = []
for raw_col, score_map, score_name in [
    ('dw_wall_mat',  WALL_SCORE,  'wall_quality_score'),
    ('dw_roof_mat',  ROOF_SCORE,  'roof_quality_score'),
    ('dw_floor_mat', FLOOR_SCORE, 'floor_quality_score'),
]:
    if raw_col in df.columns:
        df[score_name] = df[raw_col].map(score_map)
        cov = df[score_name].notna().mean()
        print(f'{score_name}: mean={df[score_name].mean():.3f}, coverage={cov:.1%}')
        _comp_scores.append(score_name)

if len(_comp_scores) >= 2:
    df['obj_quality_score'] = df[_comp_scores].mean(axis=1)
    print(f'obj_quality_score: mean={df["obj_quality_score"].mean():.3f}, '
          f'missing={df["obj_quality_score"].isna().sum()}')


In [ ]:
# ── 6.5  Crowding index ──────────────────────────────────────────────────────
if {'hh_size', 'dw_rooms'}.issubset(df.columns):
    df['persons_per_room'] = (df['hh_size'] / df['dw_rooms'].replace(0, np.nan)).clip(upper=15)
    df['is_overcrowded'] = (df['persons_per_room'] > 2).astype(int)
    print(f'persons_per_room: median={df["persons_per_room"].median():.2f}')
    print(f'is_overcrowded (>2/room): {df["is_overcrowded"].sum():,} '
          f'({df["is_overcrowded"].mean():.1%})')


In [ ]:
# ── 6.6  Dwelling age ────────────────────────────────────────────────────────
if 'dwelling_yr_built' in df.columns:
    # Consistency guard first
    bad = ~df['dwelling_yr_built'].between(1900, 2024) & df['dwelling_yr_built'].notna()
    df.loc[bad, 'dwelling_yr_built'] = np.nan
    df['dw_age_yrs'] = (2024 - df['dwelling_yr_built']).clip(0, 124)
    print(f'dw_age_yrs: median={df["dw_age_yrs"].median():.0f}yr, '
          f'missing={df["dw_age_yrs"].isna().sum()}')


In [ ]:
# ── 6.7  Triple-exposure flag ─────────────────────────────────────────────────
# triple_exposed = flood AND informal structure AND no formal title
# This compound profile identifies the highest-risk households invisible to
# any single-indicator screening.
conds = []
if 'flood_exposure' in df.columns:
    conds.append(df['flood_exposure'].gt(0).fillna(False))
if 'dw_type' in df.columns:
    conds.append(df['dw_type'].ne(1).fillna(True))   # non-permanent structure
if 'has_title_doc' in df.columns:
    conds.append(df['has_title_doc'].ne(1).fillna(True))
elif 'lp_has_title' in df.columns:
    conds.append(df['lp_has_title'].ne(1).fillna(True))

if len(conds) == 3:
    df['triple_exposed'] = (conds[0] & conds[1] & conds[2]).astype(int)
    rate = df['triple_exposed'].mean()
    print(f'triple_exposed: {df["triple_exposed"].sum():,} households ({rate:.1%})')
    print('\nTop 10 counties by triple-exposure rate:')
    if 'county_code' in df.columns:
        print(df.groupby('county_code')['triple_exposed'].mean()
               .sort_values(ascending=False).head(10)
               .rename(COUNTY_MAP).round(3).to_string())
else:
    print(f'Only {len(conds)}/3 conditions available — triple_exposed not constructed.')


In [ ]:
# ── 6.8  County name label ────────────────────────────────────────────────────
if 'county_code' in df.columns:
    df['county_name'] = df['county_code'].map(COUNTY_MAP)
    print(f'county_name: {df["county_name"].nunique()} counties, '
          f'{df["county_name"].isna().sum()} unmapped')


## 7 · Tenure-Stratified Imputation

Residual missing values (after structural encoding and engineering) are imputed
within three tenure strata — owner, renter, other. Only columns in `FINAL_47`
are imputed. **No `_imputed` flag columns are kept in the output frame** — all
imputation decisions are logged to `03_imputation_log.csv` instead.

| Miss% tier | Treatment |
|------------|-----------|
| Low (0–25%) | Group-mode (categorical) or group-median (continuous) |
| Moderate (25–60%) | Same, no flag column kept |
| High (>60%) | Left as NaN — too much assumption to impute |


In [ ]:
# ── 7.1  Build tenure strata ─────────────────────────────────────────────────
def _tenure_group(row):
    if row.get('is_renter') == 1: return 'renter'
    if row.get('is_owner')  == 1: return 'owner'
    return 'other'

df['_tenure_group'] = df[['is_renter','is_owner']].apply(_tenure_group, axis=1)     if {'is_renter','is_owner'}.issubset(df.columns) else 'all'

print('Tenure strata for imputation:')
print(df['_tenure_group'].value_counts().to_string())


In [ ]:
# ── 7.2  Classify imputation tiers (FINAL_47 columns only) ─────────────────
# Skip KES continuous columns — their NaN has domain meaning (subpopulation)
SKIP_IMPUTE = {'rent_burden_ratio', 'rent_burdened', 'util_burden_ratio',
               'total_exp',  # already engineered
               'triple_exposed', 'persons_per_room', 'is_overcrowded',
               'obj_quality_score', 'dw_age_yrs', 'is_renter', 'is_owner',
               'county_name'}

curr_miss = df[FINAL_47 & set(df.columns)].isna().mean() * 100
eligible  = curr_miss[(curr_miss > 0) & (~curr_miss.index.isin(SKIP_IMPUTE))]

LOW      = eligible[eligible <= 25].index.tolist()
MODERATE = eligible[(eligible > 25) & (eligible <= 60)].index.tolist()
HIGH     = eligible[eligible > 60].index.tolist()

print(f'LOW tier     (0–25%,  impute)  : {len(LOW)} columns')
print(f'MODERATE tier (25–60%, impute) : {len(MODERATE)} columns')
print(f'HIGH tier    (>60%, leave NaN) : {len(HIGH)} columns')
print()
if MODERATE:
    print('MODERATE-tier columns (imputed, no flag kept):')
    for c in MODERATE:
        print(f'  {c:<40}  {eligible[c]:.1f}%')
print()
if HIGH:
    print('HIGH-tier columns left as NaN:')
    for c in HIGH:
        print(f'  {c:<40}  {eligible[c]:.1f}%')


In [ ]:
# ── 7.3  Run imputation (no flag columns created) ────────────────────────────
groups = df['_tenure_group'].unique()
impute_log = []

for c in LOW + MODERATE:
    if c not in df.columns:
        continue
    pct    = float(eligible.get(c, 0))
    is_num = pd.api.types.is_numeric_dtype(df[c])

    if is_num and df[c].dropna().nunique() > 10:
        method = 'group_median'
        for g in groups:
            mask = df['_tenure_group'] == g
            med  = df.loc[mask, c].median()
            if pd.isna(med): med = df[c].median()
            df.loc[mask & df[c].isna(), c] = med
    elif is_num:
        method = 'group_mode'
        for g in groups:
            mask   = df['_tenure_group'] == g
            mode_s = df.loc[mask, c].mode(dropna=True)
            fill   = mode_s.iloc[0] if len(mode_s) else df[c].mode().iloc[0]
            df.loc[mask & df[c].isna(), c] = fill
    else:
        method = 'constant_Unknown'
        df[c] = df[c].fillna('Unknown')

    impute_log.append({'column': c, 'method': method,
                       'pct_missing_before': round(pct, 2),
                       'tier': 'Moderate' if pct > 25 else 'Low'})

df = df.drop(columns=['_tenure_group'])

impute_log_df = pd.DataFrame(impute_log)
impute_log_df.to_csv(TABS / '03_imputation_log.csv', index=False)
print(f'Imputation complete.')
print(f'  Low-tier    : {(impute_log_df["tier"]=="Low").sum()} columns')
print(f'  Moderate-tier: {(impute_log_df["tier"]=="Moderate").sum()} columns')
print(f'  Left as NaN : {len(HIGH)} columns')


## 8 · Outlier Treatment (KES & Perception Scales)

Only KES-denominated engineered monetary ratios and perception scales receive
treatment. Raw expenditure columns are transient and will be dropped.

- **Perception scale guard:** `perc_overall` must be in {1, 2, 3}.
- **Ratio caps:** `rent_burden_ratio` already clipped at 1.5 during engineering.
  `util_burden_ratio` already clipped at 1.0.


In [ ]:
# ── 8.1  Perception scale range guard ────────────────────────────────────────
if 'perc_overall' in df.columns:
    bad = ~df['perc_overall'].isin([1, 2, 3]) & df['perc_overall'].notna()
    n = bad.sum()
    if n > 0:
        df.loc[bad, 'perc_overall'] = np.nan
        print(f'perc_overall: {n} values outside {{1,2,3}} → NaN')
    else:
        print('perc_overall: all values within valid range {1,2,3}. ✓')

# ── 8.2  Consistency checks ───────────────────────────────────────────────────
# county_code validity
if 'county_code' in df.columns:
    bad = ~df['county_code'].between(1, 47) & df['county_code'].notna()
    if bad.sum():
        print(f'county_code outside 1–47: {bad.sum()} → NaN')
        df.loc[bad, 'county_code'] = np.nan

# household size sanity
if 'hh_size' in df.columns:
    bad = ~df['hh_size'].between(1, 50) & df['hh_size'].notna()
    if bad.sum(): print(f'hh_size outside 1–50: {bad.sum()} rows (review only, not changed)')

# dw_rooms sanity
if 'dw_rooms' in df.columns:
    bad = ~df['dw_rooms'].between(1, 30) & df['dw_rooms'].notna()
    if bad.sum(): print(f'dw_rooms outside 1–30: {bad.sum()} rows (review only)')

print('Consistency checks complete. ✓')


## 9 · Final Column Selection → 47-Column Analytical Frame

All transient engineering-input columns are now dropped. The output frame contains
**only** the 47 columns defined in `FINAL_47`. This is the moment where the
notebook enforces the ≤50-column budget.


In [ ]:
# ── 9.1  Drop transient columns, keep only FINAL_47 ─────────────────────────
# Also drop intermediate quality component scores (wall/roof/floor) —
# only obj_quality_score survives.
INTERMEDIATE_DROP = [
    'wall_quality_score', 'roof_quality_score', 'floor_quality_score',
    # transient engineering inputs
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
    'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
    'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
    'spend_other_kes', 'spend_remittances_kes',
    'rent_actual_kes', 'rent_monthly_kes', 'pays_rent',
    'dw_wall_mat', 'dw_roof_mat', 'dw_floor_mat',
    'dwelling_yr_built',
    'spend_water_kes', 'spend_electricity_kes',
    # individual perc_* columns (perc_overall survives)
    'perc_structure', 'perc_roof', 'perc_walls', 'perc_floor',
    'perc_ventilation', 'perc_lighting', 'perc_water', 'perc_sanitation',
    'perc_waste', 'perc_security',
    # other columns pulled in transitionally but not in FINAL_47
    'dw_area_m2', 'dw_bedrooms', 'dw_approved', 'dw_has_planning',
    'hh_tenure_type', 'tenure_is_informal', 'is_owner_occupier',
    'tenancy_duration_cat', 'year_occupied_cat', 'yrs_in_dwelling',
    'wants_to_own', 'has_written_lease', 'rent_arrears', 'rent_negotiated',
    'plans_to_buy', 'psu_min_rent_kes', 'rental_market_type',
    'dwelling_value_kes', 'plot_size_decimals',
    'lp_n_parcels', 'lp_tenure_type', 'lp_land_use', 'lp_is_registered',
    'lp_used_as_collateral',
    'owns_other_property', 'had_renovation',
    'other_hazard_exposure', 'hazard_lost_property',
    'dist_to_market_mins',
    'prob_overcrowding', 'prob_poor_water', 'prob_poor_sanitation',
    'prob_poor_drainage', 'prob_insecurity', 'prob_high_cost', 'prob_poor_structure',
    'water_collect_method', 'water_dist_mins',
    'electricity_hrs_day', 'electricity_conn_type', 'cooking_stove_type',
    'util_income_ratio', 'pays_utilities', 'cty_min_utility_kes', 'cty_med_util_ratio',
    'has_internet',     # keep only if in df; handled below
    'owns_mobile',      # same
    'n_female', 'n_children', 'n_elderly', 'n_working_age', 'mean_age',
    'cty_med_bedrooms', 'mort_rate',
    'hh_uuid', 'mortgage_repayment_kes', 'housing_cost_burden',  # handle carefully
]

# Drop what exists and is not in FINAL_47
drop_now = [c for c in df.columns if c not in FINAL_47]
df = df.drop(columns=drop_now, errors='ignore')

print(f'Columns dropped in final selection: {len(drop_now)}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
missing_from_df = FINAL_47 - set(df.columns)
if missing_from_df:
    print(f'\nNote: {len(missing_from_df)} FINAL_47 columns not present in source data:')
    for c in sorted(missing_from_df):
        print(f'  {c}')


In [ ]:
# ── 9.2  Final column inventory ──────────────────────────────────────────────
print(f'Final frame: {df.shape[0]:,} households × {df.shape[1]} columns')
print()
print('Columns present:')
for i, c in enumerate(sorted(df.columns), 1):
    miss = df[c].isna().mean() * 100
    print(f'  {i:2d}. {c:<35}  {miss:5.1f}% missing')


## 10 · Pillar Map Integrity & Anti-Leakage Assertion

All five HFVS pillars are mapped to the columns present in the final frame.
A formal assertion confirms that no pillar formula ingredient appears in the
proxy feature matrix (the anti-leakage requirement from the Business Understanding).


In [ ]:
# ── 10.1  HFVS Pillar Map (final frame columns only) ────────────────────────
PILLAR_MAP = {
    'Pillar 1 – Financial Stress': [
        'total_exp', 'rent_burden_ratio', 'rent_burdened',
        'housing_cost_burden', 'missed_payment', 'eviction_risk',
    ],
    'Pillar 2 – Physical Quality': [
        'obj_quality_score', 'perc_overall', 'dw_type', 'dw_rooms',
        'persons_per_room', 'is_overcrowded', 'dw_age_yrs',
    ],
    'Pillar 3 – Tenure Security': [
        'tenure_type', 'has_title_doc', 'owns_land',
        'lp_has_title', 'lp_has_dispute', 'satisfied_tenure',
    ],
    'Pillar 4 – Physical Hazard Exposure': [
        'flood_exposure', 'landslide_exposure', 'triple_exposed',
        'dw_in_hazard_zone', 'hazard_displaced',
    ],
    'Pillar 5 – Utility Deprivation': [
        'water_src_main', 'toilet_type', 'cooking_fuel', 'lighting_src',
        'has_handwash_facility', 'util_burden_ratio',
    ],
}

PROXY_FEATURES = {
    'hh_size', 'hh_head_sex', 'has_disability', 'max_edu_isced',
    'dependency_ratio', 'urban_rural', 'owns_mobile', 'has_internet',
    'is_renter', 'is_owner',
    'is_slum', 'settlement_plan_status', 'water_treated',
    'county_code', 'county_name',
}

print('=== Pillar map coverage ===')
all_pillar_vars = set()
for pillar, cols in PILLAR_MAP.items():
    present  = [c for c in cols if c in df.columns]
    missing_ = [c for c in cols if c not in df.columns]
    all_pillar_vars |= set(present)
    flag = f'  !! NOT IN FRAME: {missing_}' if missing_ else ''
    print(f'  {pillar:<48}  {len(present)}/{len(cols)} present{flag}')


In [ ]:
# ── 10.2  Anti-leakage assertion ─────────────────────────────────────────────
leakage = all_pillar_vars & PROXY_FEATURES
if leakage:
    print(f'LEAKAGE DETECTED in {len(leakage)} variable(s):')
    for v in sorted(leakage):
        print(f'  - {v}')
    raise AssertionError(f'{len(leakage)} variable(s) in both pillar map and proxy matrix!')
else:
    print('\nAnti-leakage assertion PASSED.')
    print('No overlap between pillar ingredients and proxy features. ✓')

# Ensure every column is accounted for (pillar OR proxy OR id/weight)
ID_WEIGHT = {'hh_id', 'hh_weight'}
accounted = all_pillar_vars | PROXY_FEATURES | ID_WEIGHT
unaccounted = set(df.columns) - accounted
if unaccounted:
    print(f'\nColumns not in any pillar, proxy, or ID set:')
    for c in sorted(unaccounted):
        print(f'  {c}')
else:
    print('Every column is assigned to a pillar, proxy set, or identity group. ✓')


## 11 · Final Audit, Visualisation & Save


In [ ]:
# ── 11.1  Final missingness summary ──────────────────────────────────────────
final_miss = (df.isna().mean() * 100).sort_values(ascending=False)
n_complete = (final_miss == 0).sum()
n_any_miss = (final_miss > 0).sum()

print(f'Final frame : {df.shape[0]:,} households × {df.shape[1]} columns')
print(f'Complete    : {n_complete} columns (0% missing)')
print(f'Any NaN     : {n_any_miss} columns')
print()
print('Columns with residual NaN:')
print(final_miss[final_miss > 0].to_string())


In [ ]:
# ── 11.2  Preprocessing summary table ────────────────────────────────────────
summary = pd.DataFrame([
    ('Raw master frame rows',       21347,         ''),
    ('Raw master frame columns',    443,           ''),
    ('Households after dedup',      df.shape[0],   '0 duplicates'),
    ('Final output columns',        df.shape[1],   'all pillar + proxy + ID'),
    ('Sentinel values decoded',     sentinel_df['n_decoded'].sum(), 'set to NaN'),
    ('Columns imputed (LOW)',        (impute_log_df['tier']=='Low').sum(),      '≤25% missing'),
    ('Columns imputed (MODERATE)',   (impute_log_df['tier']=='Moderate').sum(), '25–60% missing'),
    ('Columns left as NaN (HIGH)',   len(HIGH),     '>60% — structural or domain NaN'),
    ('Engineered features added',   9,             'total_exp, rent_burden, quality scores, etc.'),
    ('Anti-leakage assertion',      'PASSED',       'pillar ∩ proxy = ∅'),
], columns=['Metric', 'Value', 'Notes'])

summary.to_csv(TABS / '04_preprocessing_summary.csv', index=False)
print(summary.to_string(index=False))


In [ ]:
# ── 11.3  Column inventory visualisation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-column missingness in final frame
cols_sorted = final_miss[final_miss > 0].index.tolist()
pcts_sorted = final_miss[final_miss > 0].values
colors      = [RED if p > 60 else AMBER if p > 25 else BLUE for p in pcts_sorted]

axes[0].barh(cols_sorted[::-1], pcts_sorted[::-1], color=colors[::-1], edgecolor='white')
axes[0].axvline(25, color=AMBER, lw=1.2, linestyle='--', label='25% threshold')
axes[0].axvline(60, color=RED,   lw=1.2, linestyle='--', label='60% threshold')
axes[0].set_title('Residual Missingness — Final 47-Column Frame')
axes[0].set_xlabel('% missing')
axes[0].legend(fontsize=8)

# Right: pillar coverage
pillar_names   = [p.split('–')[1].strip() for p in PILLAR_MAP]
pillar_present = [sum(1 for c in v if c in df.columns) for v in PILLAR_MAP.values()]
pillar_total   = [len(v) for v in PILLAR_MAP.values()]

x = range(len(pillar_names))
axes[1].bar(x, pillar_total,   color=GRAY, alpha=0.4, label='Defined')
axes[1].bar(x, pillar_present, color=TEAL, alpha=0.9, label='In frame')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(pillar_names, rotation=20, ha='right')
axes[1].set_title('HFVS Pillar Variable Coverage')
axes[1].set_ylabel('Number of variables')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / '01_final_frame_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


In [ ]:
# ── 11.4  Pillar variable correlation heatmap ─────────────────────────────────
# Shows within-pillar and cross-pillar correlations for the numeric columns.
# This goes directly into Phase 3 planning.

numeric_cols = df.select_dtypes(include='number').drop(columns=['hh_id','hh_weight'],
                                                        errors='ignore').columns.tolist()
corr = df[numeric_cols].corr()

# Order columns by pillar
pillar_order = []
for p_cols in PILLAR_MAP.values():
    for c in p_cols:
        if c in numeric_cols and c not in pillar_order:
            pillar_order.append(c)
# Add proxy / context numeric cols at the end
for c in numeric_cols:
    if c not in pillar_order:
        pillar_order.append(c)

corr_ordered = corr.loc[pillar_order, pillar_order]

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_ordered, dtype=bool))
sns.heatmap(
    corr_ordered, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, linecolor='white',
    cbar_kws={'shrink': 0.6, 'label': 'Pearson r'},
    annot=len(corr_ordered) <= 30,  # only annotate if small enough
    fmt='.1f', annot_kws={'size': 7}
)
ax.set_title('Pairwise Correlations — HFVS Analytical Variables\n'
             '(ordered by pillar: P1→P2→P3→P4→P5→Proxy)', pad=12)
ax.tick_params(axis='both', labelsize=8)

# Add pillar boundary lines
boundaries = []
pos = 0
for p_cols in PILLAR_MAP.values():
    pos += sum(1 for c in p_cols if c in pillar_order[:pos+len(p_cols)])
    boundaries.append(pos)

for b in boundaries[:-1]:
    ax.axhline(b, color='black', lw=1.5, alpha=0.5)
    ax.axvline(b, color='black', lw=1.5, alpha=0.5)

fig.tight_layout()
fig.savefig(FIGS / '02_pillar_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation heatmap saved.')


In [ ]:
# ── 11.5  Target variable overview: which columns drive each pillar ──────────
# Print a clean pillar × variable reference table for Phase 3.
print('=' * 70)
print('HFVS PILLAR × VARIABLE REFERENCE TABLE')
print('=' * 70)
for pillar, cols in PILLAR_MAP.items():
    print(f'\n{pillar}')
    for c in cols:
        present = '✓' if c in df.columns else '✗ missing'
        miss    = f'{df[c].isna().mean()*100:.1f}% NaN' if c in df.columns else ''
        dtype   = str(df[c].dtype) if c in df.columns else ''
        print(f'  {present}  {c:<35}  {miss:<12}  {dtype}')

print()
print('PROXY FEATURES (predictors for ML classifier):')
for c in sorted(PROXY_FEATURES):
    present = '✓' if c in df.columns else '✗ missing'
    miss    = f'{df[c].isna().mean()*100:.1f}% NaN' if c in df.columns else ''
    print(f'  {present}  {c:<35}  {miss}')
print('=' * 70)


In [ ]:
# ── 11.6  Save cleaned frame ─────────────────────────────────────────────────
OUT_PATH = PQ / 'master_frame_clean.parquet'
df.to_parquet(OUT_PATH, index=False)
size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved  → {OUT_PATH}')
print(f'Size   → {size_mb:.1f} MB')
print(f'Shape  → {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Column count breakdown:')
print(f'  IDs / weights     : 4  (hh_id, county_code, county_name, hh_weight)')
print(f'  Pillar 1 – Fin.   : 6')
print(f'  Pillar 2 – Qual.  : 7')
print(f'  Pillar 3 – Tenure : 6')
print(f'  Pillar 4 – Hazard : 5')
print(f'  Pillar 5 – Utility: 6')
print(f'  Demographics/proxy: 10')
print(f'  Context           : 3')
print(f'  TOTAL             : 47  (≤ 50 budget ✓)')


---

## Outputs

| File | Location | Contents |
|------|----------|----------|
| `master_frame_clean.parquet` | `data/parquet/` | 21,347 × 47 analytical frame |
| `01_sentinel_decoding_log.csv` | `outputs/tables/dsa8301_clean/` | Cells decoded per column |
| `02_full_column_audit.csv` | `outputs/tables/dsa8301_clean/` | Pre-clean missingness inventory |
| `03_imputation_log.csv` | `outputs/tables/dsa8301_clean/` | Imputation decisions |
| `04_preprocessing_summary.csv` | `outputs/tables/dsa8301_clean/` | Pipeline summary |
| `01_final_frame_overview.png` | `outputs/figures/dsa8301_clean/` | Missingness + pillar coverage |
| `02_pillar_correlation_heatmap.png` | `outputs/figures/dsa8301_clean/` | Pairwise correlations by pillar |

**Next step → `DSA8301_Phase3_EDA.ipynb`:** load `master_frame_clean.parquet`,
run normality tests (Shapiro-Wilk, K-S), distribution plots for each pillar,
chi-square tests, and weighted county-level aggregates.

---

### Why this is 47 columns, not 528

The previous version retained **all raw inputs** alongside their derived outputs.
The key changes made here:

1. **Explicit budget constraint first.** `FINAL_47` is defined *before* any imputation or
   engineering. Every step thereafter only processes columns in that set.
2. **Transient columns.** Raw `spend_*_kes` components, individual `perc_*` ratings, and
   material codes (`dw_wall_mat` etc.) are kept only long enough to compute their
   derivatives, then dropped in Stage 9.
3. **No `_imputed` flag columns.** Imputation decisions are documented in the audit log,
   not encoded as 30+ extra binary columns in the frame.
4. **One summary per concept.** `perc_overall` replaces 11 individual perception scores;
   `obj_quality_score` replaces three material scores; `total_exp` replaces 11 spend
   components. Each concept has exactly one column in the output.
5. **Domain-informed cull in Pillar 3.** The lp_* block (7 raw columns) is reduced to
   the two most analytically discriminating: `lp_has_title` and `lp_has_dispute`.
